In [0]:
# ============================================================
# NOTEBOOK: nb_silver_po_report_dev
# Purpose : Read ekko + ekpo from Bronze, join, and MERGE
#           into po_report Delta table (SCD Type 1)
# Author  : Omkar Parab
# Version : 2.0 (Optimized)
# ============================================================

In [0]:
# ============================================================
# CELL 1: ADLS Connection + Performance Config
# ============================================================

storage_account_name = "stsapadlsrawdev"
storage_account_key  = "YOUR_STORAGE_KEY_HERE"
bronze_container     = "bronze"
silver_container     = "silver"

spark.conf.set(
    f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
    storage_account_key
)

spark.conf.set("spark.databricks.io.cache.enabled",               "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled",    "true")
spark.conf.set("spark.databricks.delta.autoCompact.enabled",      "true")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

print("ADLS connection configured!")
print("Delta Cache enabled!")
print("Auto Optimize enabled!")
print("Schema Evolution enabled!")

---------------------------------------------------------------------------
Py4JJavaError                             Traceback (most recent call last)
File <command-8574909010044296>, line 10
      7 silver_container     = "silver"
      9 # Get storage key from Key Vault (no hardcoded credentials!)
---> 10 storage_account_key = dbutils.secrets.get(
     11     scope = "kv-sap-p2p-dev",
     12     key   = "adls-storage-key"
     13 )
     15 spark.conf.set(
     16     f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net",
     17     storage_account_key
     18 )
     20 spark.conf.set("spark.databricks.io.cache.enabled",               "true")

File /databricks/python_shell/lib/dbruntime/dbutils.py:383, in DBUtils.SecretsHandler.get(self, scope, catalog, schema, key, *posArgs)
    380     scope = argsToPass[0]
    381     key = argsToPass[1]
    382     return self.entry_point.getDbutils().preview().secret(
--> 383     ).get(  # type: ignore[attr-defined]
    384      

In [0]:
# ============================================================
# CELL 2: Define Paths
# ============================================================

bronze_path    = f"abfss://{bronze_container}@{storage_account_name}.dfs.core.windows.net"
silver_path    = f"abfss://{silver_container}@{storage_account_name}.dfs.core.windows.net"

ekko_bronze    = f"{bronze_path}/ekko/"
ekpo_bronze    = f"{bronze_path}/ekpo/"
po_report_path = f"{silver_path}/po_report/"
watermark_path = f"{silver_path}/po_report_watermark/"

print(f"Bronze ekko  : {ekko_bronze}")
print(f"Bronze ekpo  : {ekpo_bronze}")
print(f"PO Report    : {po_report_path}")
print(f"Watermark    : {watermark_path}")

Bronze ekko  : abfss://bronze@stsapadlsrawdev.dfs.core.windows.net/ekko/
Bronze ekpo  : abfss://bronze@stsapadlsrawdev.dfs.core.windows.net/ekpo/
PO Report    : abfss://silver@stsapadlsrawdev.dfs.core.windows.net/po_report/
Watermark    : abfss://silver@stsapadlsrawdev.dfs.core.windows.net/po_report_watermark/


In [0]:

# ============================================================
# CELL 3: Setup Watermark Table (first run only)
# ============================================================

from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, TimestampType
from pyspark.sql.functions import col, to_timestamp, to_date, trim, \
    current_timestamp, year, month, max as spark_max, broadcast, lit

watermark_exists = DeltaTable.isDeltaTable(spark, watermark_path)

if not watermark_exists:
    schema = StructType([
        StructField("table_name",       StringType(),    False),
        StructField("last_recordstamp", TimestampType(), True)
    ])

    df_wm = spark.createDataFrame([
        ("po_report", None)
    ], schema)

    df_wm.write.format("delta").mode("overwrite").save(watermark_path)
    print("Watermark table created!")
else:
    print("Watermark table already exists!")

# Read current watermark
last_watermark = spark.read.format("delta") \
    .load(watermark_path) \
    .filter(col("table_name") == "po_report") \
    .collect()[0]["last_recordstamp"]

print(f"Current watermark: {last_watermark}")

Watermark table created!
Current watermark: None


In [0]:
# ============================================================
# CELL 4: Read ekko from Bronze (Incremental)
# ============================================================

df_ekko = spark.read \
    .format("csv") \
    .option("header",      "true") \
    .option("inferSchema", "false") \
    .load(ekko_bronze)

# Apply watermark filter
if last_watermark is not None:
    df_ekko = df_ekko.filter(
        to_timestamp(col("recordstamp")) > last_watermark
    )

print(f"ekko records to process: {df_ekko.count()}")

ekko records to process: 10


In [0]:

# ============================================================
# CELL 5: Read ekpo from Bronze (Incremental)
# ============================================================

df_ekpo = spark.read \
    .format("csv") \
    .option("header",      "true") \
    .option("inferSchema", "false") \
    .load(ekpo_bronze)

# Apply watermark filter
if last_watermark is not None:
    df_ekpo = df_ekpo.filter(
        to_timestamp(col("recordstamp")) > last_watermark
    )

print(f"ekpo records to process: {df_ekpo.count()}")

ekpo records to process: 7


In [0]:

# ============================================================
# CELL 6: Select ME2N columns from ekko and ekpo
# ============================================================

# Select only ME2N columns from ekko
df_ekko_select = df_ekko.select(
    trim(col("ebeln")).alias("po_number"),
    trim(col("lifnr")).alias("vendor_id"),
    to_date(col("bedat"), "yyyy-MM-dd").alias("po_date"),
    trim(col("statu")).alias("po_status"),
    trim(col("bsart")).alias("po_type"),
    trim(col("waers")).alias("currency"),
    trim(col("bstyp")).alias("po_category"),
    trim(col("bukrs")).alias("company_code"),
    trim(col("ekorg")).alias("purchasing_org"),
    trim(col("ekgrp")).alias("purchasing_group"),
    trim(col("frgsx")).alias("release_strategy"),
    trim(col("frgzu")).alias("release_status"),
    trim(col("frgke")).alias("release_indicator"),
    trim(col("zterm")).alias("payment_terms"),
    to_timestamp(col("recordstamp")).alias("ekko_recordstamp")
)

# Select only ME2N columns from ekpo
df_ekpo_select = df_ekpo.select(
    trim(col("ebeln")).alias("po_number"),
    trim(col("ebelp")).alias("po_line_item"),
    trim(col("matnr")).alias("material_number"),
    trim(col("txz01")).alias("material_description"),
    trim(col("meins")).alias("unit_of_measure"),
    col("menge").cast("double").alias("quantity"),
    col("netpr").cast("double").alias("net_price"),
    col("peinh").cast("double").alias("price_unit"),
    trim(col("werks")).alias("plant"),
    col("netwr").cast("double").alias("net_value"),
    trim(col("matkl")).alias("material_group"),
    trim(col("loekz")).alias("deletion_flag"),
    trim(col("pstyp")).alias("item_category"),
    trim(col("lgort")).alias("storage_location"),
    to_timestamp(col("recordstamp")).alias("ekpo_recordstamp")
)

print(f"ekko selected columns: {len(df_ekko_select.columns)}")
print(f"ekpo selected columns: {len(df_ekpo_select.columns)}")

ekko selected columns: 15
ekpo selected columns: 15


In [0]:
# ============================================================
# CELL 7: Deduplicate + Join ekko + ekpo = PO Report
# ============================================================

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

# Step 1: Deduplicate ekko - keep latest per po_number
window_ekko = Window.partitionBy("po_number") \
    .orderBy(desc("ekko_recordstamp"))

df_ekko_dedup = df_ekko_select \
    .withColumn("rn", row_number().over(window_ekko)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print(f"ekko before dedup: {df_ekko_select.count()}")
print(f"ekko after dedup : {df_ekko_dedup.count()}")

# Step 2: Deduplicate ekpo - keep latest per po_number + po_line_item
window_ekpo = Window.partitionBy("po_number", "po_line_item") \
    .orderBy(desc("ekpo_recordstamp"))

df_ekpo_dedup = df_ekpo_select \
    .withColumn("rn", row_number().over(window_ekpo)) \
    .filter(col("rn") == 1) \
    .drop("rn")

print(f"ekpo before dedup: {df_ekpo_select.count()}")
print(f"ekpo after dedup : {df_ekpo_dedup.count()}")

# Step 3: Join ekko + ekpo on po_number (broadcast small ekko table)
df_po_report = df_ekpo_dedup.join(
    broadcast(df_ekko_dedup),
    on  = "po_number",
    how = "left"
).select(
    # Header columns from ekko
    col("po_number"),
    lit(None).cast("string").alias("vendor_name"),
    col("vendor_id"),
    col("po_date"),
    col("po_status"),
    col("po_type"),
    col("currency"),
    col("po_category"),
    col("company_code"),
    col("purchasing_org"),
    col("purchasing_group"),
    col("release_strategy"),
    col("release_status"),
    col("release_indicator"),
    col("payment_terms"),

    # Line item columns from ekpo
    col("po_line_item"),
    col("material_number"),
    col("material_description"),
    col("unit_of_measure"),
    col("quantity"),
    col("net_price"),
    col("price_unit"),
    col("plant"),
    col("net_value"),
    col("material_group"),
    col("deletion_flag"),
    col("item_category"),
    col("storage_location"),

    # Derived columns
    year(col("po_date")).alias("po_year"),
    month(col("po_date")).alias("po_month"),

    # Audit columns
    col("ekpo_recordstamp").alias("source_recordstamp"),
    current_timestamp().alias("silver_load_time")
) \
.filter(col("po_number").isNotNull()) \
.filter(col("po_line_item").isNotNull())

print(f"po_report records to process: {df_po_report.count()}")
print(f"po_report columns: {len(df_po_report.columns)}")
df_po_report.show(3)


ekko before dedup: 10
ekko after dedup : 3
ekpo before dedup: 7
ekpo after dedup : 3
po_report records to process: 3
po_report columns: 32
+----------+-----------+---------+----------+---------+-------+--------+-----------+------------+--------------+----------------+----------------+--------------+-----------------+-------------+------------+---------------+--------------------+---------------+--------+---------+----------+-----+---------+--------------+-------------+-------------+----------------+-------+--------+--------------------+--------------------+
| po_number|vendor_name|vendor_id|   po_date|po_status|po_type|currency|po_category|company_code|purchasing_org|purchasing_group|release_strategy|release_status|release_indicator|payment_terms|po_line_item|material_number|material_description|unit_of_measure|quantity|net_price|price_unit|plant|net_value|material_group|deletion_flag|item_category|storage_location|po_year|po_month|  source_recordstamp|    silver_load_time|
+----------

In [0]:
# ============================================================
# CELL 8: MERGE into po_report Delta (SCD Type 1)
# ============================================================

po_report_exists = DeltaTable.isDeltaTable(spark, po_report_path)

if not po_report_exists:
    # First run → create Delta table
    df_po_report.write \
        .format("delta") \
        .mode("overwrite") \
        .partitionBy("po_year", "po_month") \
        .option("mergeSchema", "true") \
        .save(po_report_path)

    # Set table properties for optimization
    spark.sql(f"""
        ALTER TABLE delta.`{po_report_path}`
        SET TBLPROPERTIES (
            'delta.dataSkippingNumIndexedCols' = '32',
            'delta.autoOptimize.optimizeWrite' = 'true',
            'delta.autoOptimize.autoCompact' = 'true'
        )
    """)

    print("po_report Delta table created with optimizations!")

else:
    # Subsequent runs → MERGE (SCD Type 1)
    delta_po = DeltaTable.forPath(spark, po_report_path)

    delta_po.alias("target").merge(
        df_po_report.alias("source"),
        """
        target.po_number    = source.po_number AND
        target.po_line_item = source.po_line_item
        """
    ).whenMatchedUpdate(
        condition = "source.source_recordstamp > target.source_recordstamp",
        set = {
            "vendor_id"            : "source.vendor_id",
            "vendor_name"          : "source.vendor_name",
            "po_date"              : "source.po_date",
            "po_status"            : "source.po_status",
            "po_type"              : "source.po_type",
            "currency"             : "source.currency",
            "po_category"          : "source.po_category",
            "company_code"         : "source.company_code",
            "purchasing_org"       : "source.purchasing_org",
            "purchasing_group"     : "source.purchasing_group",
            "release_strategy"     : "source.release_strategy",
            "release_status"       : "source.release_status",
            "release_indicator"    : "source.release_indicator",
            "payment_terms"        : "source.payment_terms",
            "material_number"      : "source.material_number",
            "material_description" : "source.material_description",
            "unit_of_measure"      : "source.unit_of_measure",
            "quantity"             : "source.quantity",
            "net_price"            : "source.net_price",
            "price_unit"           : "source.price_unit",
            "plant"                : "source.plant",
            "net_value"            : "source.net_value",
            "material_group"       : "source.material_group",
            "deletion_flag"        : "source.deletion_flag",
            "item_category"        : "source.item_category",
            "storage_location"     : "source.storage_location",
            "source_recordstamp"   : "source.source_recordstamp",
            "silver_load_time"     : "source.silver_load_time"
        }
    ).whenNotMatchedInsertAll() \
     .execute()

    print("po_report MERGE completed!")

# Verify count
total = spark.read.format("delta").load(po_report_path).count()
print(f"Total records in po_report: {total}")

po_report MERGE completed!
Total records in po_report: 3


In [0]:

# ============================================================
# CELL 9: Update Watermark
# ============================================================

new_watermark = df_po_report \
    .agg(spark_max("source_recordstamp")) \
    .collect()[0][0]

print(f"New watermark: {new_watermark}")

if new_watermark is not None:
    delta_wm = DeltaTable.forPath(spark, watermark_path)

    delta_wm.alias("target").merge(
        spark.createDataFrame(
            [("po_report", new_watermark)],
            ["table_name", "last_recordstamp"]
        ).alias("source"),
        "target.table_name = source.table_name"
    ).whenMatchedUpdateAll() \
     .execute()

    print("Watermark updated successfully!")
else:
    print("No new records — watermark unchanged!")

# Show updated watermark
spark.read.format("delta").load(watermark_path).show()

New watermark: 2022-04-21 14:46:56.561898
Watermark updated successfully!
+----------+--------------------+
|table_name|    last_recordstamp|
+----------+--------------------+
| po_report|2022-04-21 14:46:...|
+----------+--------------------+



In [0]:
# ============================================================
# CELL 10: OPTIMIZE + Z-Order
# ============================================================

from delta.tables import DeltaTable

delta_po = DeltaTable.forPath(spark, po_report_path)

# Compact small files + Z-Order by most queried columns
delta_po.optimize().executeZOrderBy(
    "vendor_id",
    "material_number",
    "po_number"
)

print("OPTIMIZE + Z-Order completed!")

# Show Delta table history
spark.sql(f"DESCRIBE HISTORY delta.`{po_report_path}`") \
    .select("version", "timestamp", "operation", "operationMetrics") \
    .show(5, truncate=False)

OPTIMIZE + Z-Order completed!
+-------+-------------------+---------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation|operationMetrics                                                                                                                                                                                                                                                             

In [0]:
# ============================================================
# CELL 11: VACUUM
# ============================================================

# Delete old Delta versions older than 7 days
delta_po.vacuum(168)  # 168 hours = 7 days

print("VACUUM completed!")
print("Old Delta versions cleaned up!")

VACUUM completed!
Old Delta versions cleaned up!


In [0]:
# ============================================================
# CELL 12: Final Verification
# ============================================================

print("\n=== Final Summary ===")

# Watermark
print("\n-- Watermark --")
spark.read.format("delta").load(watermark_path).show()

# Record count
total = spark.read.format("delta").load(po_report_path).count()
print(f"Total records in po_report: {total}")

# Show sample data
print("\n-- Sample Data --")
spark.read.format("delta").load(po_report_path).show(5)

# Show partitions
print("\n-- Partitions --")
files = dbutils.fs.ls(
    f"abfss://silver@{storage_account_name}.dfs.core.windows.net/po_report/"
)
for f in files:
    print(f.path)


=== Final Summary ===

-- Watermark --
+----------+--------------------+
|table_name|    last_recordstamp|
+----------+--------------------+
| po_report|2022-04-21 14:46:...|
+----------+--------------------+

Total records in po_report: 3

-- Sample Data --
+----------+-----------+---------+----------+---------+-------+--------+-----------+------------+--------------+----------------+----------------+--------------+-----------------+-------------+------------+---------------+--------------------+---------------+--------+---------+----------+-----+---------+--------------+-------------+-------------+----------------+-------+--------+--------------------+--------------------+
| po_number|vendor_name|vendor_id|   po_date|po_status|po_type|currency|po_category|company_code|purchasing_org|purchasing_group|release_strategy|release_status|release_indicator|payment_terms|po_line_item|material_number|material_description|unit_of_measure|quantity|net_price|price_unit|plant|net_value|material_g